# Assignment 4
## Econ 8310 - Business Forecasting

This assignment will make use of the bayesian statistical models covered in Lessons 10 to 12. 

A/B Testing is a critical concept in data science, and for many companies one of the most relevant applications of data-driven decision-making. In order to improve product offerings, marketing campaigns, user interfaces, and many other user-facing interactions, scientists and engineers create experiments to determine the efficacy of proposed changes. Users are then randomly assigned to either the treatment or control group, and their behavior is recorded.
If the changes that the treatment group is exposed to can be measured to have a benefit in the metric of interest, then those changes are scaled up and rolled out to across all interactions.
Below is a short video detailing the A/B Testing process, in case you want to learn a bit more:
[https://youtu.be/DUNk4GPZ9bw](https://youtu.be/DUNk4GPZ9bw)

For this assignment, you will use an A/B test data set, which was pulled from the Kaggle website (https://www.kaggle.com/datasets/yufengsui/mobile-games-ab-testing). I have added the data from the page into Codio for you. It can be found in the cookie_cats.csv file in the file tree. It can also be found at [https://github.com/dustywhite7/Econ8310/raw/master/AssignmentData/cookie_cats.csv](https://github.com/dustywhite7/Econ8310/raw/master/AssignmentData/cookie_cats.csv)

The variables are defined as follows:

| Variable Name  | Definition |
|----------------|----|
| userid         | A unique number that identifies each player  |
| version        | Whether the player was put in the control group (gate_30 - a gate at level 30) or the group with the moved gate (gate_40 - a gate at level 40) |
| sum_gamerounds | The number of game rounds played by the player during the first 14 days after install.  |
| retention1     | Did the player come back and play 1 day after installing?     |
| retention7     | Did the player come back and play 7 days after installing?    |               

### The questions

You will be asked to answer the following questions in a small quiz on Canvas:
1. What was the effect of moving the gate from level 30 to level 40 on 1-day retention rates?
2. What was the effect of moving the gate from level 30 to level 40 on 7-day retention rates?
3. What was the biggest challenge for you in completing this assignment?

You will also be asked to submit a URL to your forked GitHub repository containing your code used to answer these questions.

In [4]:
#Step 1 - load & Examine data before modeling
import pandas as pd

url = "https://github.com/dustywhite7/Econ8310/raw/master/AssignmentData/cookie_cats.csv"
df = pd.read_csv(url)

# Basic exploration
print(df.shape)
print(df.head())

#Check Avg retention after 1 day and 7 days
print(df[['retention_1', 'retention_7']].mean())

(90189, 5)
   userid  version  sum_gamerounds  retention_1  retention_7
0     116  gate_30               3        False        False
1     337  gate_30              38         True        False
2     377  gate_40             165         True        False
3     483  gate_40               1        False        False
4     488  gate_40             179         True         True
retention_1    0.445210
retention_7    0.186065
dtype: float64


In [7]:
#Step 2 - Group Separation & Retention Counts
#Split into control (gate_30) and treatment (gate_40) groups
gate30 = df[df['version'] == 'gate_30']
gate40 = df[df['version'] == 'gate_40']

#1-day retention - how many players came back the next day?
r1_30_success = gate30['retention_1'].sum() #SUM counts TRUE
r1_30_total = len(gate30)                   #LEN counts all values

r1_40_success = gate40['retention_1'].sum()
r1_40_total = len(gate40)

#7-day retention - how many players came back after 1 week?
r7_30_success = gate30['retention_7'].sum()
r7_30_total = len(gate30)

r7_40_success = gate40['retention_7'].sum()
r7_40_total = len(gate40)

print("1-Day Retention:")
print(f"gate_30: {r1_30_success} retained out of {r1_30_total} ({r1_30_success/r1_30_total:.4f})")
print(f"gate_40: {r1_40_success} retained out of {r1_40_total} ({r1_40_success/r1_40_total:.4f})")

print("\n7-Day Retention:")
print(f"gate_30: {r7_30_success} retained out of {r7_30_total} ({r7_30_success/r7_30_total:.4f})")
print(f"gate_40: {r7_40_success} retained out of {r7_40_total} ({r7_40_success/r7_40_total:.4f})")

1-Day Retention:
gate_30: 20034 retained out of 44700 (0.4482)
gate_40: 20119 retained out of 45489 (0.4423)

7-Day Retention:
gate_30: 8502 retained out of 44700 (0.1902)
gate_40: 8279 retained out of 45489 (0.1820)


In [6]:
#Step 3 - Bayesian Beta-Binomial Model & Results

"""
Since retention is a binary outcome (T/F), the Beta-Binomial model should be the right Bayesian approach.
We use a Beta(1,1) uninformative prior, which is uniform, meaning we let the data do all the talking.
The posterior becomes Beta(1 + successes, 1 + failures).
We then draw samples from each group's posterior and compare them to get a probability to show which gate placement is better.
"""

import numpy as np
from scipy import stats

np.random.seed(42)
n_samples = 100000

#build posterior distributions for each group
#1-day retention posteriors
r1_30_posterior = stats.beta(1 + r1_30_success, 1 + (r1_30_total - r1_30_success))
r1_40_posterior = stats.beta(1 + r1_40_success, 1 + (r1_40_total - r1_40_success))

#7-day retention posteriors
r7_30_posterior = stats.beta(1 + r7_30_success, 1 + (r7_30_total - r7_30_success))
r7_40_posterior = stats.beta(1 + r7_40_success, 1 + (r7_40_total - r7_40_success))

#draw 100,000 samples (n_samples) from each posterior to compare the distributions
r1_30_samples = r1_30_posterior.rvs(n_samples)
r1_40_samples = r1_40_posterior.rvs(n_samples)

r7_30_samples = r7_30_posterior.rvs(n_samples)
r7_40_samples = r7_40_posterior.rvs(n_samples)

#Compare: probability that gate_30 > gate_40 aka..
#"What fraction of the time does gate_30 beat gate_40 in the samples?"
r1_prob_30_better = (r1_30_samples > r1_40_samples).mean()
r7_prob_30_better = (r7_30_samples > r7_40_samples).mean()

#Average Difference between groups
#Effect sizes (gate_40 minus gate_30)
#Negative result indicates that gate_40 performed worse
r1_effect = (r1_40_samples - r1_30_samples).mean()
r7_effect = (r7_40_samples - r7_30_samples).mean()

#Print Results
print("1-Day Retention:")
print(f"gate_30 posterior mean: {r1_30_posterior.mean():.4f}")
print(f"gate_40 posterior mean: {r1_40_posterior.mean():.4f}")
print(f"Effect of moving gate (gate_40 - gate_30): {r1_effect:+.4f} ({r1_effect*100:+.2f} percentage points)")
print(f"Probability gate_30 is better: {r1_prob_30_better:.4f} ({r1_prob_30_better*100:.1f}%)")

print("\n7-Day Retention:")
print(f"gate_30 posterior mean: {r7_30_posterior.mean():.4f}")
print(f"gate_40 posterior mean: {r7_40_posterior.mean():.4f}")
print(f"Effect of moving gate (gate_40 - gate_30): {r7_effect:+.4f} ({r7_effect*100:+.2f} percentage points)")
print(f"Probability gate_30 is better: {r7_prob_30_better:.4f} ({r7_prob_30_better*100:.1f}%)")

1-Day Retention:
gate_30 posterior mean: 0.4482
gate_40 posterior mean: 0.4423
Effect of moving gate (gate_40 - gate_30): -0.0059 (-0.59 percentage points)
Probability gate_30 is better: 0.9627 (96.3%)

7-Day Retention:
gate_30 posterior mean: 0.1902
gate_40 posterior mean: 0.1820
Effect of moving gate (gate_40 - gate_30): -0.0082 (-0.82 percentage points)
Probability gate_30 is better: 0.9992 (99.9%)
